In [5]:
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras import optimizers
from cnn_base import build_t1, build_t1_quantizedCNN

In [6]:
TRAIN_PICKLE_PATH = "./dataset/sat6_train.pkl"
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"
 
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 3e-4
BITS = 8
 
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
 
def load_pickle_data(path):
    print(f"Lendo {path} ...")
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"], data["y"]

X_train, y_train = load_pickle_data(TRAIN_PICKLE_PATH)
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)

model = build_t1_quantizedCNN(BITS)
model.summary()

Lendo ./dataset/sat6_train.pkl ...
Lendo ./dataset/sat6_test.pkl ...
Model: "t1_quantizedCNN_8b"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_layer (InputLayer)    [(None, 32, 32, 4)]       0         
                                                                 
 conv1 (QConv2D)             (None, 16, 16, 20)        720       
                                                                 
 relu1 (QActivation)         (None, 16, 16, 20)        0         
                                                                 
 conv2 (QConv2D)             (None, 16, 16, 8)         160       
                                                                 
 relu2 (QActivation)         (None, 16, 16, 8)         0         
                                                                 
 flatten (Flatten)           (None, 2048)              0         
                                             

In [7]:
optimizer = optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    X_train,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_test, y_test),
    shuffle=True,
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nAcurácia final (QKeras, 8 bits) no teste: {test_acc*100:.2f}% | loss: {test_loss:.4f}")


Epoch 1/30
633/633 [==============================] - 8s 12ms/step - loss: nan - accuracy: 0.0884 - val_loss: nan - val_accuracy: 0.0459
Epoch 2/30
633/633 [==============================] - 7s 11ms/step - loss: nan - accuracy: 0.0461 - val_loss: nan - val_accuracy: 0.0459
Epoch 3/30
633/633 [==============================] - 7s 11ms/step - loss: nan - accuracy: 0.0461 - val_loss: nan - val_accuracy: 0.0459
Epoch 4/30
633/633 [==============================] - 7s 11ms/step - loss: nan - accuracy: 0.0461 - val_loss: nan - val_accuracy: 0.0459
Epoch 5/30
633/633 [==============================] - 7s 11ms/step - loss: nan - accuracy: 0.0461 - val_loss: nan - val_accuracy: 0.0459
Epoch 6/30
633/633 [==============================] - 7s 11ms/step - loss: nan - accuracy: 0.0461 - val_loss: nan - val_accuracy: 0.0459
Epoch 7/30
633/633 [==============================] - 7s 11ms/step - loss: nan - accuracy: 0.0461 - val_loss: nan - val_accuracy: 0.0459
Epoch 8/30
633/633 [=====================

In [8]:
model.save("t1_qkeras_model.h5")
model.save_weights("t1_qkeras_weights.h5")

with open("t1_qkeras_model.json", "w") as f:
    f.write(model.to_json())